# Day 079 — Exercise 3: Executing Tools

**What you'll build:** the two functions that touch the outside world — `execute_tool` (runs a tool) and `call_llm` (calls the model).

**Why it matters:** both are **injection points**. `call_llm` takes an `llm_fn` so the whole agent can run with a mock model — that's exactly how these tests, and the gate, run with no Ollama at all.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing
    a runaway loop (a model that never says 'finish').
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── a safe calculator tool (no eval) ─────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate a basic arithmetic expression without eval().

    Supports + - * / ** % and parentheses. Anything else (names, calls,
    attribute access) raises ValueError. This is the safe way to give an
    agent a calculator: never eval() untrusted model output.
    """
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# ── the tool registry ────────────────────────────────────────────────────────
# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.
DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "word_count": {
        "description": "Count the words in a piece of text.",
        "parameters": {"text": "string - the text to count words in"},
        "fn": lambda args: str(len(str(args["text"]).split())),
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as a text block for the prompt."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)

# ── parsing messy LLM output ──────────────────────────────────────────────────
def safe_parse_json(text):
    """Extract and parse the first JSON object from messy LLM output.

    LLMs wrap JSON in markdown fences or prose. Instead of fighting that,
    slice from the first '{' to the last '}' and parse that. Returns a dict,
    or None if no valid JSON object is present.
    """
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def parse_action(text):
    """Turn raw LLM output into an action dict. NEVER raises.

    Returns one of:
      {"type": "tool",   "tool": name, "args": {...}}
      {"type": "finish", "answer": str}
    If the text is not a valid tool call, it falls back to a finish action
    holding the raw text - so a badly-formatted model reply still terminates
    the loop instead of crashing it.
    """
    data = safe_parse_json(text)
    if not isinstance(data, dict):
        return {"type": "finish", "answer": text.strip()}
    tool = data.get("tool")
    if tool and tool != "finish":
        return {"type": "tool", "tool": tool, "args": data.get("args", {})}
    return {"type": "finish", "answer": data.get("answer", text.strip())}


## Task

1. `execute_tool(action, tools) -> str` — look up `action['tool']` in `tools`. If missing, return an `Error: unknown tool ...` string. Otherwise call `tools[name]['fn'](action.get('args', {}))` inside `try/except` and return the result as a string (return the exception text on error). Must **never** raise.
2. `call_llm(messages, llm_fn=None) -> str` — if `llm_fn` is given, `return llm_fn(messages)`. Otherwise `import ollama`, `ollama.chat(model='llama3.2', messages=messages)`, return `resp['message']['content']`.

## Your Implementation

In [ ]:
def execute_tool(action, tools):
    """Run one tool from the registry. Returns a result string. Never raises."""
    raise NotImplementedError

def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str."""
    raise NotImplementedError


In [ ]:

# ── executing tools + calling the model ──────────────────────────────────────
def execute_tool(action, tools):
    """Run one tool action against the registry. Returns a result string.

    Never raises: an unknown tool or a tool error is returned as text so the
    agent can read it and recover on its next turn.
    """
    name = action.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](action.get("args", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model. Inject llm_fn(messages) -> str for testing.

    llm_fn=None uses Ollama (llama3.2). A mock llm_fn lets the whole agent
    run offline with no model - which is how the tests drive the loop.
    """
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Automated checks

In [ ]:

score, total = 0, 5
try:
    r = execute_tool({'type': 'tool', 'tool': 'calculator',
                      'args': {'expression': '2+2'}}, DEFAULT_TOOLS)
    assert r == '4'
    score += 1; print("✅ execute_tool runs a real tool")

    r2 = execute_tool({'type': 'tool', 'tool': 'nope', 'args': {}}, DEFAULT_TOOLS)
    assert 'unknown tool' in r2.lower()
    score += 1; print("✅ execute_tool reports unknown tools (no crash)")

    r3 = execute_tool({'type': 'tool', 'tool': 'calculator', 'args': {}},
                      DEFAULT_TOOLS)
    assert 'error' in r3.lower()
    score += 1; print("✅ execute_tool captures tool errors (never raises)")

    got = call_llm([{'role': 'user', 'content': 'hi'}], llm_fn=lambda m: 'MOCK')
    assert got == 'MOCK'
    score += 1; print("✅ call_llm uses the injected llm_fn")

    captured = {}
    def _spy(m):
        captured['m'] = m
        return 'ok'
    call_llm([{'role': 'user', 'content': 'x'}], llm_fn=_spy)
    assert isinstance(captured['m'], list) and captured['m'][0]['role'] == 'user'
    score += 1; print("✅ llm_fn receives the messages list")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── executing tools + calling the model ──────────────────────────────────────
def execute_tool(action, tools):
    """Run one tool action against the registry. Returns a result string.

    Never raises: an unknown tool or a tool error is returned as text so the
    agent can read it and recover on its next turn.
    """
    name = action.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](action.get("args", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model. Inject llm_fn(messages) -> str for testing.

    llm_fn=None uses Ollama (llama3.2). A mock llm_fn lets the whole agent
    run offline with no model - which is how the tests drive the loop.
    """
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]
```

**Why return errors as text instead of raising?** The agent reads the result string on its next turn. An error it can *see* (`Error running calculator: ...`) is something it can recover from; an exception just kills the loop.

</details>